In [ ]:
from google.colab import drive
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.types import DoubleType, IntegerType, FloatType
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import time

In [ ]:
color_1 = '#27AAE2'
color_2 = '#2B3990'

In [ ]:
drive.mount('/content/drive')

In [ ]:
# !pip install pyspark
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("clustering analysis") \
    .getOrCreate()

In [ ]:
# Define the input folder path on Drive
input_folder_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_pulito_finale.csv"

In [ ]:
# Load the DataFrame

df = ss.read.csv(
    input_folder_path,
    header=True,
    inferSchema=True,
    sep=","
)

print(f"Number of partitions: {df.rdd.getNumPartitions()}")
df.printSchema()
df.show(5)

In [ ]:
# Define the columns that must not be used as features for clustering
# (Coordinates will be dropped, Class and Confidence will be kept as metadata)
cols_to_exclude = ['XCoords', 'YCoords', 'Class', 'Confidence']

input_cols = [
    field.name for field in df.schema.fields
    if isinstance(field.dataType, DoubleType) and field.name not in cols_to_exclude
]

cols_to_keep = input_cols + ['Class', 'Confidence']

df_final_numeric = df.select(*cols_to_keep)

print(f"Numerical features ready for the Assembler ({len(input_cols)} columns):")
print(input_cols)

print("\nFinal schema (Numerical Features + Saved Metadata):")
df_final_numeric.printSchema()
df_final_numeric.show(5)

In [ ]:
assembler = VectorAssembler(
    inputCols=input_cols,
    outputCol="unscaled_features"
)

scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

k = 15
kmeans = KMeans(
    featuresCol="features",
    k=k,
    seed=42
)

# Definition and Training of the Pipeline
pipeline = Pipeline(stages=[assembler, scaler, kmeans])

# Apply the pipeline to the DataFrame
model = pipeline.fit(df_final_numeric)

# Get the DataFrame with assigned clusters
df_clustered = model.transform(df_final_numeric)


# Compute SSE
kmeans_model = model.stages[-1]
costo_wssse = kmeans_model.summary.trainingCost
print(f"SSE: {costo_wssse}")

# Show the first rows to confirm that we have both 'prediction' and 'Class'
df_clustered.select("Class", "Confidence", "prediction", "features").show(5, truncate=False)

In [ ]:
# Extract the trained KMeans model (the last stage of the Pipeline)
kmeans_model = model.stages[-1]

# centroids
centers = kmeans_model.clusterCenters()

feature_names = input_cols

# Convert the centroids array into a Pandas DataFrame for visualization
# Each row is a cluster, each column is a feature
df_centroids = pd.DataFrame(
    centers,
    columns=feature_names
)

# Add cluster id
df_centroids['Cluster'] = [f"Cluster {i+1}" for i in range(k)]

In [ ]:
# Convert the Pandas DataFrame from "wide" format to "long" format
# This is necessary for libraries like Seaborn or Altair
df_long = pd.melt(
    df_centroids,
    id_vars=['Cluster'],
    var_name='Features',
    value_name='Centroid_Values'
)

print(df_long.head())

In [ ]:
# Parallel coordinate plot
fig = px.line(
    df_long,
    x='Features',
    y='Centroid_Values',
    color='Cluster',
    markers=True,
    template='plotly_white'
)

fig.update_xaxes(tickangle=45, showgrid=True, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridcolor='LightGray')

fig.update_layout(yaxis_title="Standardized Centroids Values",
                  font=dict(size=22),
                  legend=dict(font=dict(size=15)),
                  margin=dict(t=40, b=80, l=80, r=40),
                  paper_bgcolor='white',
                  plot_bgcolor='white')

fig.show()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_clustered_with_class = df_clustered.select("Class", "prediction")

df_clustered_with_class.show(5)

In [ ]:
# Contingency Table
df_counts = df_clustered_with_class.groupBy("Class", "prediction").count()

window_by_class = Window.partitionBy("Class")
window_by_cluster = Window.partitionBy("prediction")

df_analysis = df_counts.withColumn(
    "Total_Class",
    F.sum("count").over(window_by_class)
).withColumn(
    "Total_Cluster",
    F.sum("count").over(window_by_cluster)
)

df_results = df_analysis.withColumn(
    # Percentage of Class in the Cluster: "For each class, what percentage of the total for that class is here?"
    "P_of_Total_Class",
    (F.col("count") / F.col("Total_Class") * 100).cast("double")
).withColumn(
    # Percentage of Class in the Cluster: "For this cluster, what percentage of the elements belong to this class?"
    "P_of_Cluster_Total",
    (F.col("count") / F.col("Total_Cluster") * 100).cast("double")
).select(
    "Class",
    F.col("prediction").alias("Cluster"),
    F.col("count").alias("Count_in_Cluster"),
    "Total_Class",
    "Total_Cluster",
    F.round("P_of_Total_Class", 2).alias("P_of_Total_Class (%)"),
    F.round("P_of_Cluster_Total", 2).alias("P_of_Cluster_Total (%)")
).orderBy("Cluster", F.col("P_of_Cluster_Total (%)").desc())

df_results.show(300, truncate=False)

In [ ]:
# SSE
kmeans_model = model.stages[-1]
costo_wssse = kmeans_model.summary.trainingCost

# Silhouette Score
evaluator_global = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette"
)

silhouette_score = evaluator_global.evaluate(df_clustered)

print(f"SSE: {costo_wssse:.2f}")
print(f"Silhouette Score: {silhouette_score:.4f}")

df_clustered.select("Class", "Confidence", "prediction", "features").show(5, truncate=False)

# OPTIMIZED K-MEANS

In [ ]:
print("SSE and Silhouette Score for K 2-20")

k_values = range(2, 21)
wssse_values = []
silhouette_values = []

df_elbow_input = df_clustered.drop("prediction")
df_elbow_input.cache()
df_elbow_input.count()

start_time = time.time()

for k in k_values:
    kmeans_elbow = KMeans(featuresCol="features", k=k, seed=42)
    model_elbow = kmeans_elbow.fit(df_elbow_input)


    cost = model_elbow.summary.trainingCost
    wssse_values.append(cost)

    predictions = model_elbow.transform(df_elbow_input)
    silhouette = evaluator_global.evaluate(predictions)
    silhouette_values.append(silhouette)

    print(f"K={k} done. SSE: {cost:.2f} | Silhouette: {silhouette:.4f}")

end_time = time.time()
print(f"\n{end_time - start_time:.2f} seconds!")

In [ ]:
df_metrics = pd.DataFrame({
    'K': list(k_values),
    'WSSSE': wssse_values,
    'Silhouette': silhouette_values
})

fig = make_subplots(
    rows=1, cols=2
)

fig.add_trace(
    go.Scatter(x=df_metrics['K'], y=df_metrics['WSSSE'], mode='lines+markers', name='WSSSE', marker=dict(color='#2B3990')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df_metrics['K'], y=df_metrics['Silhouette'], mode='lines+markers', name='Silhouette', marker=dict(color='#27AAE2')),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of Clusters (K)", row=1, col=1, showgrid=True, gridcolor='LightGray')
fig.update_yaxes(title_text="SSE", row=1, col=1, showgrid=True, gridcolor='LightGray')

fig.update_xaxes(title_text="Number of Clusters (K)", row=1, col=2, showgrid=True, gridcolor='LightGray')
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2, showgrid=True, gridcolor='LightGray')

fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=500,
    font=dict(size=22),
    margin=dict(t=40, b=80, l=80, r=30),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.update_xaxes(
    tickmode='linear',
    tick0=1,
    dtick=1,
    tickangle=0,
    tickfont=dict(size=18)
)

fig.show()

In [ ]:
# Final model training (K=6)
k_ottimale = 6
print(f"Training final model with K={k_ottimale}")

# Remove the old 'prediction' column (from the initial K=15) so as not to confuse Spark
df_pulito = df_clustered.drop("prediction")

kmeans_finale = KMeans(featuresCol="features", k=k_ottimale, seed=42)
modello_finale = kmeans_finale.fit(df_pulito)

# Overwrite df_clustered: this way the subsequent PCA charts will use K=6!
df_clustered = modello_finale.transform(df_pulito)

centroidi_finali = modello_finale.clusterCenters()
df_centroidi_finali = pd.DataFrame(centroidi_finali, columns=input_cols)
df_centroidi_finali['Cluster'] = [f"Cluster {i}" for i in range(k_ottimale)]

df_counts_finale = df_clustered.groupBy("Class", "prediction").count()

window_class = Window.partitionBy("Class")
window_cluster = Window.partitionBy("prediction")

df_results_finale = df_counts_finale.withColumn(
    "Total_Class", F.sum("count").over(window_class)
).withColumn(
    "Total_Cluster", F.sum("count").over(window_cluster)
).withColumn(
    "P_of_Total_Class (%)", F.round((F.col("count") / F.col("Total_Class") * 100), 2)
).withColumn(
    "P_of_Cluster_Total (%)", F.round((F.col("count") / F.col("Total_Cluster") * 100), 2)
).select(
    "Class",
    F.col("prediction").alias("Cluster"),
    F.col("count").alias("Count_in_Cluster"),
    "Total_Class",
    "Total_Cluster",
    "P_of_Total_Class (%)",
    "P_of_Cluster_Total (%)"
).orderBy("Cluster", F.col("P_of_Cluster_Total (%)").desc())

df_results_finale.show(100, truncate=False)

In [ ]:
sse_final = modello_finale.summary.trainingCost
silhouette_final = evaluator_global.evaluate(df_clustered)

print(f"Training of the final model completed K={k_ottimale}.")
print(f"SSE: {sse_final:.2f}")
print(f"Silhouette Score: {silhouette_final:.4f}")

In [ ]:
target_df = df_clustered

total_rows = target_df.count()

df_distribution = target_df.groupBy("prediction").count().withColumn(
    "Percentage",
    (F.col("count") / total_rows * 100)
).select(
    F.col("prediction").alias("Cluster"),
    F.col("count").alias("Num_Punti"),
    F.round("Percentage", 2).alias("Percentuale (%)")
)

df_distribution_sorted = df_distribution.orderBy(F.col("Num_Punti").desc())

print(f"Total rows: {total_rows} ---")
print("Clusters distribution:")
df_distribution_sorted.show(30, truncate=False)

In [ ]:
pca = PCA(k=2, inputCol="features", outputCol="pcaFeatures")

pca_model = pca.fit(df_clustered)

df_pca = pca_model.transform(df_clustered)

var_spiegata = pca_model.explainedVariance
print(f"Explained variance PC1: {var_spiegata[0]*100:.2f}%")
print(f"Explained variance PC2: {var_spiegata[1]*100:.2f}%")
print(f"Total preserved variance: {(sum(var_spiegata)*100):.2f}%")

In [ ]:
FRACTION = 0.2  #Percentage for the subsampling
SEED = 42

df_viz_spark = df_pca.select("pcaFeatures", "prediction").sample(
    withReplacement=False,
    fraction=FRACTION,
    seed=SEED
)

df_viz_pandas = df_viz_spark.toPandas()

print(f"\nNumber of points extracted for the plot: {len(df_viz_pandas)}")

In [ ]:
# We need to split pca features into two separate columns 'PC1' and 'PC2'.

df_viz_pandas["PC1"] = df_viz_pandas["pcaFeatures"].apply(lambda x: x[0])
df_viz_pandas["PC2"] = df_viz_pandas["pcaFeatures"].apply(lambda x: x[1])

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df_viz_pandas,
    x="PC1",
    y="PC2",
    hue="prediction",
    palette="tab10",
    s=30,
    alpha=0.6,
    style="prediction"
)

plt.xlabel(f'Principal Component 1', fontsize=22)
plt.ylabel(f'Principal Component 2', fontsize=22)
plt.legend(title='Cluster ID', bbox_to_anchor=(1.05, 1), loc='upper left', title_fontsize=18, fontsize=16)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.tight_layout()
plt.show()

In [ ]:
# Convert in long format
df_long_finale = pd.melt(
    df_centroidi_finali,
    id_vars=['Cluster'],
    var_name='Features',
    value_name='Centroid_Values'
)

# Parallel line plot
fig_k6 = px.line(
    df_long_finale,
    x='Features',
    y='Centroid_Values',
    color='Cluster',
    markers=True,
    template='plotly_white'
)

fig_k6.update_layout(
    xaxis_title="Features",
    yaxis_title="Standardized Centroids Values",
    legend_font_size=18,
    height=600,
    margin=dict(t=40, b=80, l=80, r=40),
    font=dict(size=22),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig_k6.update_xaxes(tickangle=45, showgrid=True, gridcolor='LightGray')
fig_k6.update_yaxes(showgrid=True, gridcolor='LightGray')

fig_k6.show()

In [ ]:
print(df_centroidi_finali.head())

## SUBCLUSTERING ON MARINE DEBRIS

In [ ]:
# Filter the original dataframe to isolate only "Marine Debris"
df_debris_full = df.filter(F.col("Class") == "Marine Debris")

df_debris_features = df_debris_full.select(*input_cols)

df_debris_features.cache()
total_debris = df_debris_features.count()

print(f"Total Marine Debris pixels isolated: {total_debris}")

In [ ]:
assembler_sub = VectorAssembler(
    inputCols=input_cols,
    outputCol="unscaled_features_sub"
)

scaler_sub = StandardScaler(
    inputCol="unscaled_features_sub",
    outputCol="features_sub",
    withStd=True,
    withMean=True
)

k_values_sub = range(2, 10)
wssse_values_sub = []
silhouette_values_sub = []

print(f"SSE and Silhouette calculation for Debris sub-clustering (K from {min(k_values_sub)} to {max(k_values_sub)})...")

evaluator_sub = ClusteringEvaluator(
    predictionCol="sub_prediction",
    featuresCol="features_sub",
    metricName="silhouette"
)

for k in k_values_sub:
    # Add 'predictionCol="sub_prediction"' not overwriting the previous prediction
    kmeans_sub_loop = KMeans(featuresCol="features_sub", predictionCol="sub_prediction", k=k, seed=42)

    pipeline_sub_loop = Pipeline(stages=[assembler_sub, scaler_sub, kmeans_sub_loop])

    model_sub_loop = pipeline_sub_loop.fit(df_debris_features)

    # SSE
    cost = model_sub_loop.stages[-1].summary.trainingCost
    wssse_values_sub.append(cost)

    # Silhouette
    predictions_sub_loop = model_sub_loop.transform(df_debris_features)
    silhouette = evaluator_sub.evaluate(predictions_sub_loop)
    silhouette_values_sub.append(silhouette)

    print(f"K={k} completed. SSE: {cost:.2f} | Silhouette: {silhouette:.4f}")

In [ ]:
df_metrics_sub = pd.DataFrame({
    'K': list(k_values_sub),
    'WSSSE': wssse_values_sub,
    'Silhouette': silhouette_values_sub
})

fig = make_subplots(
    rows=1, cols=2
)

fig.add_trace(
    go.Scatter(x=df_metrics_sub['K'], y=df_metrics_sub['WSSSE'], mode='lines+markers', name='WSSSE', marker=dict(color='#2B3990')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df_metrics_sub['K'], y=df_metrics_sub['Silhouette'], mode='lines+markers', name='Silhouette', marker=dict(color='#27AAE2')),
    row=1, col=2
)

fig.update_xaxes(title_text="Number of Clusters (K)", row=1, col=1, showgrid=True, gridcolor='LightGray')
fig.update_yaxes(title_text="SSE", row=1, col=1, showgrid=True, gridcolor='LightGray')

fig.update_xaxes(title_text="Number of Clusters (K)", row=1, col=2, showgrid=True, gridcolor='LightGray')
fig.update_yaxes(title_text="Silhouette Score", row=1, col=2, showgrid=True, gridcolor='LightGray')

fig.update_xaxes(tickmode='linear', tick0=2, dtick=1)
fig.update_layout(
    template='plotly_white',
    showlegend=False,
    height=500,
    font=dict(size=22),
    margin=dict(t=40, b=80, l=80, r=20),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.show()

In [ ]:
# Train final model
k_sub = 4

kmeans_sub = KMeans(
    featuresCol="features_sub",
    predictionCol="sub_prediction",
    k=k_sub,
    seed=42
)

pipeline_sub = Pipeline(stages=[assembler_sub, scaler_sub, kmeans_sub])

# We fit and transform on the FULL debris dataframe to keep metadata like 'Confidence'
model_sub = pipeline_sub.fit(df_debris_full)
df_debris_clustered = model_sub.transform(df_debris_full)

In [ ]:
sse_sub = model_sub.stages[-1].summary.trainingCost

silhouette_final_sub = evaluator_sub.evaluate(df_debris_clustered)

print(f"SSE: {sse_sub:.2f}")
print(f"Silhouette Score: {silhouette_final_sub:.4f}")

In [ ]:
kmeans_sub_model = model_sub.stages[-1]
centers_sub = kmeans_sub_model.clusterCenters()

df_centroids_sub = pd.DataFrame(
    centers_sub,
    columns=input_cols
)

df_centroids_sub['Sub_Cluster'] = [f"Sub-Cluster {i}" for i in range(k_sub)]

df_long_sub = pd.melt(
    df_centroids_sub,
    id_vars=['Sub_Cluster'],
    var_name='Features',
    value_name='Centroid_Values'
)

fig_sub_cent = px.line(
    df_long_sub,
    x='Features',
    y='Centroid_Values',
    color='Sub_Cluster',
    markers=True,
    template='plotly_white'
)

fig_sub_cent.update_xaxes(tickangle=45, showgrid=True, gridcolor='LightGray')
fig_sub_cent.update_yaxes(showgrid=True, gridcolor='LightGray')

fig_sub_cent.update_layout(yaxis_title="Standardized Centroids Values",
                           margin=dict(t=40, b=80, l=80, r=40),
                           font=dict(size=22),
                           paper_bgcolor='white',
                           plot_bgcolor='white')
fig_sub_cent.show()

In [ ]:
print("Centroids values:")
print(df_centroids_sub)

for i, row in df_centroids_sub.iterrows():
    print(f"\n--- {row['Sub_Cluster']} ---")
    for feature, value in row.drop('Sub_Cluster').items():
        print(f"{feature}: {value:.4f}")

In [ ]:
# Analyze Sub-Cluster Composition against Annotator Confidence

df_confidence_analysis = df_debris_clustered.groupBy("sub_prediction", "Confidence").count()

# Calculate percentages within each sub-cluster
from pyspark.sql.window import Window

window_sub_cluster = Window.partitionBy("sub_prediction")

df_confidence_results = df_confidence_analysis.withColumn(
    "Total_in_SubCluster",
    F.sum("count").over(window_sub_cluster)
).withColumn(
    "Confidence_Percentage",
    F.round((F.col("count") / F.col("Total_in_SubCluster") * 100), 2)
).select(
    F.col("sub_prediction").alias("Sub_Cluster_ID"),
    "Confidence",
    F.col("count").alias("Num_Pixels"),
    "Confidence_Percentage"
).orderBy("Sub_Cluster_ID", F.col("Confidence_Percentage").desc())

print("\nAnnotator Confidence distribution within Marine Debris Sub-Clusters:")
df_confidence_results.show(truncate=False)

In [ ]:
# Subclustering PCA visualization
pca_sub = PCA(k=2, inputCol="features_sub", outputCol="pcaFeatures_sub")
pca_model_sub = pca_sub.fit(df_debris_clustered)
df_pca_sub = pca_model_sub.transform(df_debris_clustered)

var_spiegata_sub = pca_model_sub.explainedVariance
print(f"Variance PC1: {var_spiegata_sub[0]*100:.2f}%")
print(f"Variance PC2: {var_spiegata_sub[1]*100:.2f}%")
print(f"Total variance preserved: {(sum(var_spiegata_sub)*100):.2f}%")

df_viz_spark_sub = df_pca_sub.select("pcaFeatures_sub", "sub_prediction")
df_viz_pandas_sub = df_viz_spark_sub.toPandas()

print(f"\nTotal number of points for the subcluster: {len(df_viz_pandas_sub)}")

# Components separation
df_viz_pandas_sub["PC1"] = df_viz_pandas_sub["pcaFeatures_sub"].apply(lambda x: x[0])
df_viz_pandas_sub["PC2"] = df_viz_pandas_sub["pcaFeatures_sub"].apply(lambda x: x[1])

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df_viz_pandas_sub,
    x="PC1",
    y="PC2",
    hue="sub_prediction",
    palette="tab10",
    s=60,
    alpha=0.8,
    style="sub_prediction"
)

plt.xlabel(f'Principal Component 1 ({var_spiegata_sub[0]*100:.1f}%)', fontsize=22)
plt.ylabel(f'Principal Component 2 ({var_spiegata_sub[1]*100:.1f}%)', fontsize=22)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.legend(
    title='Cluster ID',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    fontsize=16,
    title_fontsize=18
)

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()